In [4]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 24.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
import scipy
from tqdm import tqdm

In [2]:
with open('parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [15]:
sam_cj.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [24]:
for item in sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique():
    if item[-2:] == 'NN':
        parent_dict[item] = 'not hypo'
    if item[:2] == 'mo':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mg':
        parent_dict[item] = 'hypo'
    if item[:2] == 'xt':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ac':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cj':
        parent_dict[item] = 'hypo'
    if item[:2] == 'dr':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cc':
        parent_dict[item] = 'hypo'
    if item[:2] == 'rv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
        
parent_dict['not hypo'] = 'not hypo'
parent_dict['hypo'] = 'hypo'
parent_dict['Unknown'] = 'hypo'
parent_dict['Unknwon_2'] = 'hypo'
parent_dict['Non neuron'] = 'not hypo'

In [4]:
Fullorthotable = pd.read_csv('OTO_star_nothreshold_6species_ohnlogs_expressionthresh_08022026.tsv',delimiter='\t', index_col = 'MM')

In [7]:
sam_mg = SAM()
sam_mg.load_data('Active_SAM_joined/SAM_Allen_Institute_allhypo_01272026.h5ad')
gene_dict_mg = {}
for i in range(len(sam_mg.adata.var_names)):
    gene_dict_mg[sam_mg.adata.var_names[i]] = i

In [8]:
sam_mo = SAM()
sam_mo.load_data('Active_SAM_joined/SAM_MO_soupx_plus5_cleaned_NN_04142026.h5ad')
gene_dict_mo = {}
for i in range(len(sam_mo.adata.var_names)):
    gene_dict_mo[sam_mo.adata.var_names[i]] = i

In [9]:
sam_cj = SAM()
sam_cj.load_data('Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad')
gene_dict_cj = {}
for i in range(len(sam_cj.adata.var_names)):
    gene_dict_cj[sam_cj.adata.var_names[i]] = i

In [10]:
sam_ac= SAM()
sam_ac.load_data('Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad')
gene_dict_ac = {}
for i in range(len(sam_ac.adata.var_names)):
    gene_dict_ac[sam_ac.adata.var_names[i]] = i

In [11]:
sam_xt = SAM()
sam_xt.load_data('Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad')
gene_dict_xt = {}
for i in range(len(sam_xt.adata.var_names)):
    gene_dict_xt[sam_xt.adata.var_names[i]] = i

In [12]:
sam_dr = SAM()
sam_dr.load_data('Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad')
gene_dict_dr = {}
for i in range(len(sam_dr.adata.var_names)):
    gene_dict_dr[sam_dr.adata.var_names[i]] = i

In [20]:
TF = pd.read_csv('Fin_TF_06012026.csv')
TF_set = set(TF['0'])

In [21]:
overall_TF_set = set(Fullorthotable.index) & TF_set

In [22]:
mg_TF_cl = [item for item in overall_TF_set]
mo_TF_cl = [Fullorthotable.loc[item,'MO'] for item in overall_TF_set]
cj_TF_cl = [Fullorthotable.loc[item,'CJ'] for item in overall_TF_set]
ac_TF_cl = [Fullorthotable.loc[item,'AC'] for item in overall_TF_set]
xt_TF_cl = [Fullorthotable.loc[item,'XT'] for item in overall_TF_set]
dr_TF_cl = [Fullorthotable.loc[item,'DR'] for item in overall_TF_set]

In [37]:
hypo_xt = [parent_dict[parent_dict[i]] == 'hypo' for i in sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn']]

In [38]:
np.sort(sam_xt.adata[hypo_xt].obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique())

array(['073 MEA-BST Sox6 Gaba', '074 MEA-BST Lhx6 Sp9 Gaba',
       '079 CEA-BST Six3 Cyp26b1 Gaba', '082 CEA-BST Ebf1 Pdyn Gaba',
       '085 SI-MPO-LPO Lhx8 Gaba', '086 MPO-ADP Lhx8 Gaba',
       '088 BST Tac2 Gaba', '090 BST-MPN Six3 Nrgn Gaba',
       '091 ARH-PVi Six6 Dopa-Gaba', '092 TMv-PMv Tbx3 Hist-Gaba',
       '093 RT-ZI Gnb3 Gaba', '094 SCH Six6 Cdc14a Gaba',
       '097 PVHd-SBPV Six3 Prox1 Gaba', '098 AHN-SBPV-PVHd Pdrm12 Gaba',
       '100 AHN Onecut3 Gaba', '101 ZI Pax6 Gaba',
       '102 DMH-LHA Gsx1 Gaba', '103 PVHd-DMH Lhx6 Gaba',
       '104 TU-ARH Otp Six6 Gaba', '105 TMd-DMH Foxd2 Gaba',
       '106 PVpo-VMPO-MPN Hmx2 Gaba', '107 DMH Hmx2 Gaba',
       '108 ARH-PVp Tbx3 Gaba', '111 TRS-BAC Sln Glut',
       '115 MS-SF Bsx Glut', '118 ADP-MPO Trp73 Glut',
       '119 SI-MA-LPO-LHA Skor1 Glut', '122 LHA-MEA Otp Glut',
       '124 MPN-MPO-PVpo Hmx2 Glut', '125 DMH Hmx2 Glut',
       '126 ARH-PVp Tbx3 Glut', '127 DMH-LHA Vgll2 Glut',
       '129 VMH Nr5a1 Glut', '130 

In [39]:
missing_mg = []
exp = sam_mg.adata.X.A
for item in mg_TF_cl:
    if item in gene_dict_mg.keys():
        ind = gene_dict_mg[item]
        if np.sum(exp[:,ind]) > 0:
            missing_mg.append(False)
        else:
            missing_mg.append(True)
    else:
        missing_mg.append(True)
print('fin mg')
        
missing_mo = []
hypo_mo = [parent_dict[parent_dict[i]] == 'hypo' for i in sam_mo.adata.obs['ss_subclass_nounlabeled_03102026']]
exp_mo = sam_mo.adata.X.A[hypo_mo,:]
for item in mo_TF_cl:
    if item in gene_dict_mo.keys():
        ind = gene_dict_mo[item]
        if np.sum(exp_mo[:,ind]) > 0:
            missing_mo.append(False)
        else:
            missing_mo.append(True)
    else:
        missing_mo.append(True)
print('fin mo')
        
missing_cj = []
hypo_cj = [parent_dict[parent_dict[i]] == 'hypo' for i in sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn']]
exp_cj = sam_cj.adata.X.A[hypo_cj,:]
for item in cj_TF_cl:
    if item in gene_dict_cj.keys():
        ind = gene_dict_cj[item]
        if np.sum(exp_cj[:,ind]) > 0:
            missing_cj.append(False)
        else:
            missing_cj.append(True)
    else:
        missing_cj.append(True)
print('fin cj')
        
missing_ac = []
hypo_ac = [parent_dict[parent_dict[i]] == 'hypo' for i in sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn']]
exp_ac = sam_ac.adata.X.A[hypo_ac,:]
for item in ac_TF_cl:
    if item in gene_dict_ac.keys():
        ind = gene_dict_ac[item]
        if np.sum(exp_ac[:,ind]) > 0:
            missing_ac.append(False)
        else:
            missing_ac.append(True)
    else:
        missing_ac.append(True)
print('fin ac')
        
missing_xt = []
hypo_xt = [parent_dict[parent_dict[i]] == 'hypo' for i in sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn']]
exp_xt = sam_xt.adata.X.A[hypo_xt,:]
for item in xt_TF_cl:
    if item in gene_dict_xt.keys():
        ind = gene_dict_xt[item]
        if np.sum(exp_xt[:,ind]) > 0:
            missing_xt.append(False)
        else:
            missing_xt.append(True)
    else:
        missing_xt.append(True)
print('fin xt')

missing_dr = []
hypo_dr = [parent_dict[parent_dict[i]] == 'hypo' for i in sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn']]
exp_dr = sam_dr.adata.X.A[hypo_dr,:]
for item in dr_TF_cl:
    if item in gene_dict_dr.keys():
        ind = gene_dict_dr[item]
        if np.sum(exp_dr[:,ind]) > 0:
            missing_dr.append(False)
        else:
            missing_dr.append(True)
    else:
        missing_dr.append(True)
print('fin dr')

fin mg
fin mo
fin cj
fin ac
fin xt
fin dr


In [40]:
fin_ind = []
for i in range(len(missing_mg)):
    if int(missing_mg[i]) + int(missing_cj[i]) + int(missing_ac[i]) + int(missing_mo[i]) + int(missing_xt[i]) + int(missing_dr[i]) == 0:
        fin_ind.append(i)

In [41]:
len(fin_ind)

454

In [42]:
# Compute TF sparsity (reference distribution for expression matching)
# Fraction of cells with nonzero expression, averaged across all 6 species,
# for each of the 415 TFs that passed the expression filter.

tf_sparsity = np.zeros(len(fin_ind))
for k, i in enumerate(fin_ind):
    s_mg = np.mean(exp[:, gene_dict_mg[mg_TF_cl[i]]] > 0)
    s_mo = np.mean(exp_mo[:, gene_dict_mo[mo_TF_cl[i]]] > 0)
    s_cj = np.mean(exp_cj[:, gene_dict_cj[cj_TF_cl[i]]] > 0)
    s_ac = np.mean(exp_ac[:, gene_dict_ac[ac_TF_cl[i]]] > 0)
    s_xt = np.mean(exp_xt[:, gene_dict_xt[xt_TF_cl[i]]] > 0)
    s_dr = np.mean(exp_dr[:, gene_dict_dr[dr_TF_cl[i]]] > 0)
    tf_sparsity[k] = np.mean([s_mg, s_mo, s_cj, s_ac, s_xt, s_dr])

print(f'TF sparsity: mean={tf_sparsity.mean():.3f}, '
      f'min={tf_sparsity.min():.3f}, max={tf_sparsity.max():.3f}')

TF sparsity: mean=0.084, min=0.000, max=0.547


In [43]:
# Build candidate gene pool
# All genes in the orthotable that are:
#   - not in the TF set
#   - expressed (sum > 0) in all 6 species
#   - have a single unambiguous ortholog name in every species
#     (comma-separated DR entries like "lmx1ba,lmx1bb" fail the dict lookup
#      and are naturally excluded, matching the behaviour in the main analysis)

pool_data = {'MG': [], 'MO': [], 'CJ': [], 'AC': [], 'XT': [], 'DR': [],
             'sparsity': []}

for mg_gene in tqdm(Fullorthotable.index):
    if mg_gene in overall_TF_set:
        continue

    row = Fullorthotable.loc[mg_gene]
    mo_gene = row['MO']
    cj_gene = row['CJ']
    ac_gene = row['AC']
    xt_gene = row['XT']
    dr_gene = row['DR']

    checks = [
        (mg_gene, gene_dict_mg, exp),
        (mo_gene, gene_dict_mo, exp_mo),
        (cj_gene, gene_dict_cj, exp_cj),
        (ac_gene, gene_dict_ac, exp_ac),
        (xt_gene, gene_dict_xt, exp_xt),
        (dr_gene, gene_dict_dr, exp_dr),
    ]

    sparsities = []
    valid = True
    for gname, gdict, gexp in checks:
        if gname not in gdict:
            valid = False
            break
        ind = gdict[gname]
        if np.sum(gexp[:, ind]) == 0:
            valid = False
            break
        sparsities.append(float(np.mean(gexp[:, ind] > 0)))

    if valid:
        pool_data['MG'].append(mg_gene)
        pool_data['MO'].append(mo_gene)
        pool_data['CJ'].append(cj_gene)
        pool_data['AC'].append(ac_gene)
        pool_data['XT'].append(xt_gene)
        pool_data['DR'].append(dr_gene)
        pool_data['sparsity'].append(np.mean(sparsities))

pool_df = pd.DataFrame(pool_data).reset_index(drop=True)
print(f'Candidate pool: {len(pool_df)} genes')

100%|█████████████████████████████████████| 10189/10189 [03:40<00:00, 46.19it/s]

Candidate pool: 7845 genes


In [44]:
# Assign sparsity bins
# Use TF sparsity quantiles as bin edges so every bin has roughly equal TF
# representation. Pool genes are placed into the same bins.

N_BINS = 10
bin_edges = np.percentile(tf_sparsity, np.linspace(0, 100, N_BINS + 1))
bin_edges[0] = 0.0
bin_edges[-1] = 1.0 + 1e-9          # ensure the max TF falls inside

tf_bins  = np.digitize(tf_sparsity,           bin_edges[1:])  # 0 ... N_BINS-1
pool_bins = np.digitize(pool_df['sparsity'].values, bin_edges[1:])
pool_df['sparsity_bin'] = pool_bins

print('TF counts per bin:  ', np.bincount(tf_bins,  minlength=N_BINS))
print('Pool counts per bin:', np.bincount(pool_bins, minlength=N_BINS))

TF counts per bin:   [46 45 45 46 45 45 46 45 45 46]
Pool counts per bin: [ 424  483  381  732  621  901 1051  895 1264 1093]


In [45]:
sam_cj.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [51]:
# Helper functions

def num_coexpressed_fast(array, threshold):
    """Vectorised replacement for the original nested-loop version."""
    return int(np.triu(array > threshold, k=1).sum())


def avg_by_celltype(exp_mat, gene_inds, obs_series):
    exp_mini = exp_mat[:, gene_inds]
    cell_types = obs_series.unique()
    exp_ct = np.zeros((len(cell_types), len(gene_inds)))
    for i, ct in enumerate(cell_types):
        exp_ct[i, :] = np.average(exp_mini[obs_series == ct, :], axis=0)
    return exp_ct


def compute_corr_pearson(exp_ct):
    """Pearson correlation matrix via np.corrcoef; NaN -> 0 for zero-var genes."""
    corr = np.corrcoef(exp_ct.T)
    corr = np.nan_to_num(corr, nan=0.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def compute_corr_spearman(exp_ct):
    """Spearman correlation matrix (rank-transform each gene, then Pearson)."""
    ranked = np.apply_along_axis(stats.rankdata, 0, exp_ct)
    return compute_corr_pearson(ranked)


def run_pipeline(mg_genes, mo_genes, cj_genes, ac_genes, xt_genes, dr_genes):
    """
    Given matched gene lists (one per species), compute cell-type-averaged
    expression, per-species correlation matrices, and the leave-one-out
    conservation score for both Pearson and Spearman.
    Returns (fin_pearson, fin_spearman).
    """
    inds_mg = [gene_dict_mg[g] for g in mg_genes]
    inds_mo = [gene_dict_mo[g] for g in mo_genes]
    inds_cj = [gene_dict_cj[g] for g in cj_genes]
    inds_ac = [gene_dict_ac[g] for g in ac_genes]
    inds_xt = [gene_dict_xt[g] for g in xt_genes]
    inds_dr = [gene_dict_dr[g] for g in dr_genes]

    exp_ct_mg = avg_by_celltype(exp,    inds_mg, sam_mg.adata.obs['subclass_id_label'])
    exp_ct_mo = avg_by_celltype(exp_mo, inds_mo, sam_mo.adata.obs['ss_subclass_nounlabeled_03102026'][hypo_mo])
    exp_ct_cj = avg_by_celltype(exp_cj, inds_cj, sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'][hypo_cj])
    exp_ct_ac = avg_by_celltype(exp_ac, inds_ac, sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'][hypo_ac])
    exp_ct_xt = avg_by_celltype(exp_xt, inds_xt, sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'][hypo_xt])
    exp_ct_dr = avg_by_celltype(exp_dr, inds_dr, sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'][hypo_dr])

    results = {}
    for method, fn in [('pearson', compute_corr_pearson)]:
        c_mg = fn(exp_ct_mg)
        c_mo = fn(exp_ct_mo)
        c_cj = fn(exp_ct_cj)
        c_ac = fn(exp_ct_ac)
        c_xt = fn(exp_ct_xt)
        c_dr = fn(exp_ct_dr)

        results[method] = np.maximum.reduce([
            np.minimum.reduce([c_mo, c_cj, c_ac, c_xt, c_dr]),
            np.minimum.reduce([c_mg, c_cj, c_ac, c_xt, c_dr]),
            np.minimum.reduce([c_mg, c_mo, c_ac, c_xt, c_dr]),
            np.minimum.reduce([c_mg, c_mo, c_cj, c_xt, c_dr]),
            np.minimum.reduce([c_mg, c_mo, c_cj, c_ac, c_dr]),
            np.minimum.reduce([c_mg, c_mo, c_cj, c_ac, c_xt]),
        ])

    return results['pearson']


def sample_sparsity_matched(pool_df, tf_bins, rng):
    """
    For each TF sparsity bin, draw the same number of random genes from that
    bin in the pool. Falls back to sampling with replacement if a bin is
    under-populated (rare edge case).
    """
    sampled_idx = []
    for b in range(N_BINS):
        n_needed = int(np.sum(tf_bins == b))
        if n_needed == 0:
            continue
        available = pool_df.index[pool_df['sparsity_bin'] == b].tolist()
        if len(available) == 0:
            available = pool_df.index.tolist()
        replace = len(available) < n_needed
        chosen = rng.choice(available, size=n_needed, replace=replace)
        sampled_idx.extend(chosen.tolist())
    return pool_df.loc[sampled_idx].reset_index(drop=True)

In [52]:
# Run null iterations

N_ITER = 300
thresholds = [.25,.3,.35,.4]
null_counts = {thr: [] for thr in thresholds}

rng = np.random.default_rng(42)

for _ in tqdm(range(N_ITER), desc='null iterations'):
    sampled = sample_sparsity_matched(pool_df, tf_bins, rng)
    fin_null = run_pipeline(
        sampled['MG'].tolist(), sampled['MO'].tolist(), sampled['CJ'].tolist(),
        sampled['AC'].tolist(), sampled['XT'].tolist(), sampled['DR'].tolist(),
    )
    for thr in thresholds:
        null_counts[thr].append(num_coexpressed_fast(fin_null, thr))

null iterations: 100%|████████████████████████| 300/300 [31:26<00:00,  6.29s/it]


In [53]:
# Results table
# Compare observed TF counts against the empirical null distribution.
# fin_array_raw must already be computed from the main analysis above.

null_results = []
for thr in thresholds:
    null_arr = np.array(null_counts[thr])
    null_mean = null_arr.mean()
    null_std  = null_arr.std()
    null_p95  = np.percentile(null_arr, 95)
    null_p05  = np.percentile(null_arr, 5)
    null_results.append({
        'Threshold':      thr,
        'Null_mean':      null_mean,
        'Null_std':       null_std,
        'Null_95th':      null_p95,
        'Null_5th': null_p05,
    })

null_results_df = pd.DataFrame(null_results)
null_results_df.to_csv('TF_pearsoncorr_null_08042026.csv')
print(null_results_df)

   Threshold    Null_mean    Null_std  Null_95th  Null_5th
0       0.25  1023.040000  130.335254    1227.10    816.85
1       0.30   345.950000   56.717259     440.00    255.95
2       0.35    96.920000   22.138058     135.05     64.00
3       0.40    22.336667    7.495109      37.00     12.00
